# DualGNN NTFE demo

Sample a few 4D NTFEs of the h11=86 reflexive polytope by combining
per-2-face dualGNN draws and the [NTFE algorithm](https://arxiv.org/abs/2309.10855).

Accept rate via the NTFE algorithm is intrinsic to the polytope (~0.4% for h11=86), so even
a small `N` may take a minute or two; bump `n_workers` to parallelize.

In [ ]:
from pathlib import Path

import numpy as np
import torch
from cytools import Polytope

import dualgnn
from dualgnn.device import default_device
from dualgnn.model import DualGNN
from dualgnn.ntfe  import sample_ntfes

## Config

In [ ]:
CKPT   = Path(dualgnn.__file__).resolve().parents[2] / "ckpts" / "reinforce.pt"
DEVICE = default_device()

# h11=86 polytope
verts = [[-1, -1, -1, -1], [-1, -1, -1, 3], [-1, -1, 3, -1],
         [-1, 3, -1, -1], [1, -1, -1, -1], [1, -1, -1, 3],
         [1, -1, 3, -1], [1, 3, -1, -1]]
poly = Polytope(np.asarray(verts, dtype=np.int64))

print(f"h11 = {poly.h11()}, n_pts = {poly.points().shape[0]}, "
      f"n_2faces = {len(poly.faces(2))}")

## Load the model

In [ ]:
net = DualGNN.from_ckpt(CKPT, DEVICE)
print(f"model D={net.D} K={net.K} "
      f"params={sum(p.numel() for p in net.parameters()):,} device={DEVICE}")

## Sample

`sample_ntfes` returns either heights (default) or cytools `Triangulation`
objects (`as_triangs=True`).

In [ ]:
heights = sample_ntfes(poly, net, N=20, N_face_triangs=1_000, seed=0, n_workers=-1)
print("heights:", heights.shape, heights.dtype)
heights